<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/transpose_shared.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:


from numba.core.types import float32
from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32

@cuda.jit
def transpose (input_mat, output_mat):
  n,m = output_mat.shape

  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

  if row < n and col < m:
    output_mat[row,col] = input_mat[col,row]



def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1, 1, size=(m, n), dtype=cp.float32)
  output_mat = cp.zeros((n, m), dtype= cp.float32)

  #launch configuration
  # no. of elements in smem = no. of threads in block
  threads_per_block = (TILE_DIM, TILE_DIM)
  blocks_per_grid = (
      (m + (TILE_DIM - 1)) // TILE_DIM,
      (n + (TILE_DIM - 1)) // TILE_DIM
  )

  transpose[blocks_per_grid,threads_per_block] (input_mat,output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)

  np.testing.assert_allclose(output_mat_np, input_mat_np.T)

  print ("input matrix")
  print (input_mat_np)
  print ("output matrix")
  print (output_mat_np)

main()

input matrix
[[ 0.6274344  -0.7078728   0.85301864 ... -0.92146486  0.31995964
   0.10753953]
 [ 0.5722946   0.8125     -0.1869874  ... -0.10614002  0.4440416
  -0.6349838 ]
 [ 0.4853542   0.7125124   0.4107889  ... -0.8529893  -0.19529492
  -0.3460511 ]
 ...
 [-0.8416783  -0.33530563  0.754472   ... -0.79783314 -0.6785128
   0.73861957]
 [ 0.21972644  0.24928021 -0.6664187  ...  0.554613    0.21879816
   0.80637205]
 [ 0.17033327 -0.93492293  0.668031   ... -0.57505834  0.6778636
   0.7335881 ]]
output matrix
[[ 0.6274344   0.5722946   0.4853542  ... -0.8416783   0.21972644
   0.17033327]
 [-0.7078728   0.8125      0.7125124  ... -0.33530563  0.24928021
  -0.93492293]
 [ 0.85301864 -0.1869874   0.4107889  ...  0.754472   -0.6664187
   0.668031  ]
 ...
 [-0.92146486 -0.10614002 -0.8529893  ... -0.79783314  0.554613
  -0.57505834]
 [ 0.31995964  0.4440416  -0.19529492 ... -0.6785128   0.21879816
   0.6778636 ]
 [ 0.10753953 -0.6349838  -0.3460511  ...  0.73861957  0.80637205
   0.733588

In [13]:
%%writefile transpose-naive.py



from numba.core.types import float32
from numba import cuda
import cupy as cp
import numpy as np

TILE_DIM = 32

@cuda.jit
def transpose (input_mat, output_mat):
  n,m = output_mat.shape

  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y

  if row < n and col < m:
    output_mat[row,col] = input_mat[col,row]



def main():

  m = 10000
  n = 2000

  device = cp.cuda.Device()

  input_mat = cp.random.uniform(-1, 1, size=(m, n), dtype=cp.float32)
  output_mat = cp.zeros((n, m), dtype= cp.float32)

  #launch configuration
  # no. of elements in smem = no. of threads in block
  threads_per_block = (TILE_DIM, TILE_DIM)
  blocks_per_grid = (
      (m + (TILE_DIM - 1)) // TILE_DIM,
      (n + (TILE_DIM - 1)) // TILE_DIM
  )

  transpose[blocks_per_grid,threads_per_block] (input_mat,output_mat)

  cuda.synchronize()

  input_mat_np = cp.asnumpy(input_mat)
  output_mat_np = cp.asnumpy(output_mat)

  np.testing.assert_allclose(output_mat_np, input_mat_np.T)

  print ("input matrix")
  print (input_mat_np)
  print ("output matrix")
  print (output_mat_np)

main()

Overwriting transpose-naive.py


In [14]:
!ncu --set full -f -o transpose-naive-rep python3 transpose-naive.py

==PROF== Connected to process 11890 (/usr/bin/python3.13)
==PROF== Profiling "generate_seed_pseudo" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "gen_sequenced" - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_random_x_mod_1" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_scale" - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "transpose[abi:v1,cw51cXTLSUwv..." - 4: 0%....50%....100% - 33 passes
input matrix
[[-0.89589566 -0.40623707 -0.41444367 ... -0.63192904 -0.81280017
  -0.61681825]
 [ 0.4804622  -0.9444792   0.40663815 ...  0.8287686  -0.86456215
   0.7280222 ]
 [-0.7823479   0.05308604 -0.00146037 ...  0.55108714  0.86406374
  -0.3279791 ]
 ...
 [-0.97985834 -0.36396408  0.6862813  ...  0.7931515  -0.43349975
  -0.7967312 ]
 [ 0.18044221 -0.66039145  0.9821969  ... -0.9645709   0.09791994
   0.5359075 ]
 [-0.62312305  0.934448    0.9787854  ... -0.26886517 -0.2072388
  -0.73191285]]
output matrix
[[-0.89589566  0.4804622  -0.7823479  ... -0.